# AgentCore Memory를 사용하는 Strands Agents (도구를 통한 장기 메모리)

## 소개

이 Notebook에서는 AgentCore Memory와 Strands 프레임워크를 사용하여 **장기 메모리를 공유하는 멀티 에이전트 시스템**을 구현하는 방법을 보여 줍니다. 여러 전문 에이전트가 각자 전용 네임스페이스를 통해 공통 장기 메모리 저장소에 접근하면서 협업하는 방식을 살펴봅니다.

### 튜토리얼 세부 정보

| 항목                 | 세부 정보                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 장기 대화형                                                                       |
| 에이전트 유형       | 여행 예약 도우미                                                                  |
| 에이전트 프레임워크 | Strands Agents                                                                   |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | AgentCore 사용자 선호도 메모리 추출, 메모리 저장 및 검색 도구                                     |
| 예제 난이도         | 중급                                                                              |

이 튜토리얼에서 학습할 내용은 다음과 같습니다.

- 장기 메모리 전략을 적용한 공유 메모리 리소스 설정 방법
- 자체 메모리 네임스페이스에 접근하는 전문 에이전트 생성 방법
- 전문 에이전트에 작업을 위임하는 조정 에이전트 구현 방법
- 구조화된 메모리 네임스페이스를 활용해 에이전트를 전문화하는 방법

## 시나리오 배경

이 예제에서는 다음 구성 요소로 이루어진 **여행 계획 시스템**을 만듭니다.
1. 여행 선호도와 이력을 장기 메모리에 저장하는 항공편 예약 도우미
2. 숙박 선호도를 장기 메모리에 저장하는 호텔 예약 도우미
3. 이러한 전문 에이전트를 조율하는 여행 조정 에이전트

각 전문 에이전트는 공통 메모리 저장소 안의 자체 네임스페이스에 접근하여 시간이 지나도 사용자 선호도를 지속적으로 파악할 수 있습니다. 이 접근 방식은 복잡한 도메인을 전문 에이전트로 나누면서 메모리 인프라는 공유하고 각자의 전문 영역은 유지하는 방법을 보여 줍니다.

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
- Python 3.10+
- Amazon Bedrock AgentCore Memory 권한이 있는 AWS 자격 증명
- Amazon Bedrock AgentCore SDK

환경을 설정하고 공유 장기 메모리 리소스를 생성해 보겠습니다.

## 1단계: 환경 설정
먼저 이 Notebook을 실행하는 데 필요한 라이브러리를 가져오고 클라이언트를 정의합니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
import time
from datetime import datetime
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("travel-assistant")

Amazon Bedrock 모델과 AgentCore에 필요한 권한이 설정된 리전을 지정합니다.

In [ ]:
region = "us-west-2"  # 사용할 AWS 리전으로 변경합니다.
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

## 2단계: 공유 메모리 리소스 생성
이 섹션에서는 각 에이전트의 전용 네임스페이스를 갖춘 공통 장기 메모리 저장소를 생성합니다.

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

client = MemoryClient(region_name=region)
memory_name = "TravelBookingMemory"

In [ ]:
print("Creating or retrieving Memory with Long-Term Strategy...")
memory = client.create_or_get_memory(
    name=memory_name,
    description="Travel Agent with Long-Term Memory",
    strategies=[
        {
            StrategyType.USER_PREFERENCE.value: {
                "name": "UserPreferences",
                "description": "Captures user preferences",
                "namespaceTemplates": ["/travel/{actorId}/preferences/"],
            }
        }
    ],
    event_expiry_days=7,  # 단기 대화는 7일 후 만료됩니다.
)

memory_id = memory["id"]
print(f"✅ Memory ready: {memory_id}")

### 장기 메모리 전략 이해

생성할 메모리 리소스는 하나의 사용자 선호도 전략과 함께 AgentCore Memory의 장기 메모리 기능을 사용합니다.

1. **사용자 선호도 메모리 전략**: 대화에서 언급된 사용자 선호도를 자동으로 추출하고 통합합니다.
2. **`actorId` 기반 네임스페이스**: 동일한 사용자 `actorId` 아래에서 에이전트 도메인별 하위 네임스페이스(항공편/호텔)를 사용합니다.
3. **메모리 지속성**: 만료되는 단기 메모리와 달리, 추출된 선호도는 대화가 만료된 후에도 유지됩니다.

네임스페이스 패턴 `travel/{actorId}/flight/preferences/ and travel/{actorId}/hotel/preferences/`을 사용하면 각 전문 에이전트는 `actorId`를 기반으로 고유한 네임스페이스를 갖습니다.
- 항공편 에이전트 접근 경로: `/travel/flight-user-TIMESTAMP/preferences/`
- 호텔 에이전트 접근 경로: `/travel/hotel-user-TIMESTAMP/preferences/`

따라서 각 에이전트는 공통 메모리 인프라를 사용하면서도 자체 전문 지식을 유지할 수 있습니다.

### 에이전트 ID 설정

In [ ]:
# actorId는 USER ID를 나타내며, 메모리 지속성을 위해 세션 간 동일하게 유지합니다.
user_actor_id = "user-001"
session_id = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"

# 두 에이전트는 동일한 네임스페이스를 공유하며, 시맨틱 검색으로 항공편과 호텔 선호도를 구분합니다.
flight_actor_id = user_actor_id
hotel_actor_id = user_actor_id
flight_namespace = f"/travel/{user_actor_id}/preferences/"
hotel_namespace = f"/travel/{user_actor_id}/preferences/"

In [ ]:
# 필요한 구성 요소를 가져옵니다.
from strands import Agent, tool
from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider

### 3단계: Memory Hook Provider 생성

이 단계에서는 메모리 작업을 자동화하는 사용자 정의 `MemoryHookProvider` 클래스를 정의합니다. Hook은 에이전트 실행 수명 주기의 특정 시점에 실행되는 특수 함수입니다. 여기서 생성하는 Memory Hook의 주요 기능은 다음과 같습니다.

1. **메모리 저장**: 에이전트가 응답한 후 새 대화를 저장합니다.

이를 통해 수동 관리 없이 원활한 메모리 환경을 구현할 수 있습니다.

In [ ]:
class MemoryHookProvider(HookProvider):
    """메모리를 자동 관리하는 훅 제공자입니다."""

    def __init__(self, memory_id: str, client: MemoryClient):
        self.memory_id = memory_id
        self.client = client

    def save_memories(self, event: AfterInvocationEvent):
        """에이전트 응답 후 대화를 저장합니다."""
        try:
            messages = event.agent.messages
            if len(messages) >= 2:
                # 마지막 user 및 assistant 메시지를 가져옵니다.
                user_msg = None
                assistant_msg = None

                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not assistant_msg:
                        assistant_msg = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_msg and "toolResult" not in msg["content"][0]:
                        user_msg = msg["content"][0]["text"]
                        break

                if user_msg and assistant_msg:
                    # 에이전트 상태에서 세션 정보를 가져옵니다.
                    actor_id = event.agent.state.get("actor_id")
                    session_id = event.agent.state.get("session_id")

                    if not actor_id or not session_id:
                        logger.warning("Missing actor_id or session_id in agent state")
                        return

                    # 대화를 저장합니다.
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=actor_id,
                        session_id=session_id,
                        messages=[(user_msg, "USER"), (assistant_msg, "ASSISTANT")],
                    )
                    logger.info("Saved conversation to memory")

        except Exception as e:
            logger.error(f"Failed to save memories: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        """메모리 훅을 등록합니다."""
        registry.add_callback(AfterInvocationEvent, self.save_memories)
        logger.info("Memory hooks registered")

### 메모리에 접근하는 전문 에이전트 생성

다음으로 전문 에이전트의 시스템 프롬프트를 정의합니다.

In [ ]:
# 호텔 예약 전문 에이전트의 시스템 프롬프트
HOTEL_BOOKING_PROMPT = """You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner.Keep the messages short, don't overwhelm the customer."""

# 항공편 예약 전문 에이전트의 시스템 프롬프트
FLIGHT_BOOKING_PROMPT = """You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner.Keep the messages short, don't overwhelm the customer."""

### 에이전트 도구 구현
이제 전문 에이전트를 조정 에이전트가 사용할 수 있는 도구로 구현합니다.

In [ ]:
@tool
def flight_booking_assistant(query: str) -> str:
    """
    Process and respond to flight booking queries.

    Args:
        query: A flight-related question about bookings, schedules, airlines, or travel policies

    Returns:
        Detailed flight information, booking options, or travel advice
    """
    try:
        provider_flight = AgentCoreMemoryToolProvider(
            memory_id=memory_id,  # 필수
            actor_id=flight_actor_id,  # 필수
            session_id=session_id,  # 필수
            region=region,
            namespace=flight_namespace,
        )

        flight_memory_hooks = MemoryHookProvider(memory_id, client)

        flight_agent = Agent(
            tools=provider_flight.tools,
            hooks=[flight_memory_hooks],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT,
            state={"actor_id": flight_actor_id, "session_id": session_id},
        )

        # 에이전트를 호출하고 응답을 반환합니다.
        response = flight_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in flight booking assistant: {str(e)}"


@tool
def hotel_booking_assistant(query: str) -> str:
    """
    Process and respond to hotel booking queries.

    Args:
        query: A hotel-related question about accommodations, amenities, or reservations

    Returns:
        Detailed hotel information, booking options, or accommodation advice
    """
    try:
        provider_hotel = AgentCoreMemoryToolProvider(
            memory_id=memory_id,
            actor_id=hotel_actor_id,
            session_id=session_id,
            region=region,
            namespace=hotel_namespace,
        )

        hotel_memory_hooks = MemoryHookProvider(memory_id, client)

        hotel_booking_agent = Agent(
            tools=provider_hotel.tools,
            hooks=[hotel_memory_hooks],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT,
            state={"actor_id": hotel_actor_id, "session_id": session_id},
        )

        # 에이전트를 호출하고 응답을 반환합니다.
        response = hotel_booking_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in hotel booking assistant: {str(e)}"

### 조정 에이전트 생성

마지막으로 이러한 전문 도구를 조율하는 기본 여행 계획 에이전트를 생성합니다.

In [ ]:
# 조정 에이전트의 시스템 프롬프트
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_assistant tool
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_assistant tool
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Keep the messages short, don't overwhelm the customer.
"""

In [ ]:
travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant],
)

## 멀티 에이전트 메모리 시스템 테스트

여행 계획 시나리오로 멀티 에이전트 시스템을 테스트해 보겠습니다.

### 선택 사항: 항공편 예약 도우미의 장기 메모리에 데이터 채우기
항공편 예약 에이전트의 메모리에 데이터를 미리 채우려면 다음 셀의 주석을 해제하세요.

In [ ]:
"""flight_previous_messages = [
    ("Hi, I'm Sarah", "USER"),
    ("Hello Sarah! Welcome to FlightAssist. How can I help you with your travel plans today?", "ASSISTANT"),
    ("I'm looking to book a flight from New York to London sometime next month.", "USER"),
    ("I'd be happy to help you find flights from New York to London for next month. Do you have specific dates in mind, or are you flexible?", "ASSISTANT"),
    ("I'm thinking around the 15th to the 25th, but I can be a bit flexible.", "USER"),
    ("Great! That gives us some room to find the best options. Do you have any preferences regarding airlines or flight times?", "ASSISTANT"),
    ("I definitely prefer direct flights if possible. I really don't like layovers.", "USER"),
    ("I completely understand your preference for direct flights. Layovers can be inconvenient. Fortunately, there are several airlines offering direct flights between New York and London, including British Airways, American Airlines, Delta, and Virgin Atlantic.", "ASSISTANT"),
    ("That's good to hear. I've had good experiences with British Airways in the past.", "USER"),
    ("British Airways does offer excellent service on transatlantic routes. I'll keep that in mind when searching for options. Do you have any seating preferences or other requirements for your flight?", "ASSISTANT"),
    ("I always try to get an aisle seat. I like being able to get up without disturbing others, especially on long flights.", "USER"),
    ("An aisle seat is a great choice for long-haul flights like New York to London. I'll note your preference for aisle seating. Would you prefer to fly in the morning, afternoon, or evening?", "ASSISTANT"),
    ("I prefer overnight flights for long journeys. It helps me adjust to the time difference better.", "USER"),
    ("Overnight flights are indeed a smart choice for eastbound transatlantic travel. They allow you to arrive in London in the morning and help minimize jet lag. British Airways, Delta, and American all offer evening departures from New York that arrive in London the next morning.", "ASSISTANT"),
    ("Perfect! And I'm also wondering about baggage allowances since I'll be staying for about a week.", "USER"),
    ("For a week-long trip, most travelers find that a standard checked bag plus a carry-on is sufficient. British Airways typically allows one free checked bag on transatlantic flights in economy class, plus a carry-on and personal item. Would you like me to check the specific allowances for your preferred dates?", "ASSISTANT")
]

print("\nHydrating memories with previous conversations...")

# Save the conversation history to short-term memory
initial = client.create_event(
    memory_id=memory_id,
    actor_id=flight_actor_id,
    session_id=session_id,
    messages=flight_previous_messages,
)
print("✓ Conversation saved in short term memory")"""

In [ ]:
travel_agent("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

In [ ]:
travel_agent("I prefer direct flights with Iberia")

In [ ]:
travel_agent("I would like a flight in the morning, in economy")

In [ ]:
travel_agent("I would like to fly from SNA, and return 15 days later")

## 메모리 지속성 테스트

메모리 시스템이 올바르게 작동하는지 테스트하기 위해 여행 에이전트의 새 인스턴스를 생성하고 이전에 저장된 정보에 접근할 수 있는지 확인합니다.

In [ ]:
time.sleep(60)  # 메모리가 이벤트를 처리할 시간을 줍니다.
# 여행 에이전트의 새 인스턴스를 생성합니다.
new_travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant],
)

# 이전 대화에 관해 질문합니다.
new_travel_agent("Can you remind me about flight preferences?")

## 요약

이 Notebook에서는 다음 내용을 살펴봤습니다.

1. 여러 에이전트가 사용할 공유 메모리 리소스를 생성하는 방법
2. 메모리에 접근하는 전문 에이전트를 도구로 구현하는 방법
3. 대화 컨텍스트를 유지하면서 여러 에이전트를 조율하는 방법
4. 서로 다른 에이전트 인스턴스 간에 메모리가 지속되는 방식

공유 메모리를 사용하는 이 멀티 에이전트 아키텍처는 일관된 사용자 경험을 유지하면서 전문 도메인을 처리하는 복잡한 대화형 AI 시스템을 구축하는 효과적인 접근 방식입니다.

## 리소스 정리
이 Notebook에서 사용한 리소스를 정리하기 위해 메모리를 삭제합니다.

In [ ]:
# client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
# )